# 02 - Publish capture to SeaweedFS and apply SCD Type 1

The default `CAPTURE_MODE = "replay"` reads the supplied captured-binlog directory. Run this notebook directly without MySQL. For your own capture, select `CAPTURE_MODE = "live"` and use its database name. Use the Spark-enabled kernel in OpenShift. This notebook includes its complete configuration, upload logic and transformations. Polaris stores the Iceberg catalog; SeaweedFS stores raw batches and Iceberg files.

Use the same database name in all three notebooks. Start a fresh kernel, run every cell in order, then stop Spark before opening notebook 03.


In [ ]:
import os
import re
from pathlib import Path

DB = os.getenv("SCD_DATABASE", "scd_workshop")
assert re.fullmatch(r"[a-z][a-z0-9_]{0,40}", DB)
CAPTURE_MODE = "replay"  # Use "live" after capturing your own source changes.
assert CAPTURE_MODE in ("replay", "live")
CAPTURE_ROOT = Path.cwd() / ("captured-binlog" if CAPTURE_MODE == "replay" else "lab_data") / DB
LAB_DIR = Path(os.getenv("SCD_CAPTURE_DIR", str(CAPTURE_ROOT)))
LAB_DIR.mkdir(parents=True, exist_ok=True)
BRONZE = f"s3a://warehouse/scd-capture/{DB}"
WAREHOUSE = f"s3://warehouse/iceberg/{DB}"
print("Database / namespace:", DB)
print("Captured batches:", LAB_DIR / "batches")


## Start Spark with Polaris and SeaweedFS

Use the existing credential environment variables and runtime JARs. No download or local helper module is needed for Spark setup.

In [ ]:
import os

import socket

REQUIRED_ENV = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "POLARIS_CLIENT_ID", "POLARIS_CLIENT_SECRET"]

def preflight():
    missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing existing credential environment names: " + ", ".join(missing))
    import importlib.util
    from pathlib import Path
    if importlib.util.find_spec("pyspark") is None:
        raise RuntimeError("Choose the existing Spark-enabled Python kernel; do not pip install PySpark")
    jars = Path(os.environ.get("SPARK_HOME", "/opt/spark")) / "jars"
    patterns = ["iceberg-spark-runtime-*.jar", "iceberg-aws-bundle-*.jar",
                "hadoop-aws-*.jar", "aws-java-sdk-bundle-*.jar"]
    absent = [pattern for pattern in patterns if not list(jars.glob(pattern))]
    if absent:
        raise RuntimeError("Missing image JARs; stop without rebuilding/installing: " + ", ".join(absent))
    print("PREFLIGHT_PASS: required environment names, PySpark and image JARs present")

def start_spark(app_name):
    preflight()
    from pyspark.sql import SparkSession
    if SparkSession.getActiveSession() is not None:
        raise RuntimeError("Stop the existing Spark session and restart this kernel before setup")
    builder = (SparkSession.builder.appName(app_name).master("spark://spark-master:7077")
        .config("spark.driver.host", os.environ.get("POD_IP") or socket.gethostbyname(socket.gethostname()))
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.cores.max", "1").config("spark.executor.cores", "1")
        .config("spark.executor.memory", "512m").config("spark.sql.shuffle.partitions", "2")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.polaris.type", "rest")
        .config("spark.sql.catalog.polaris.uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog"))
        .config("spark.sql.catalog.polaris.warehouse", "training")
        .config("spark.sql.catalog.polaris.rest.auth.type", "oauth2")
        .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")
        .config("spark.sql.catalog.polaris.oauth2-server-uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog").rstrip("/") + "/v1/oauth/tokens")
        .config("spark.sql.catalog.polaris.header.Polaris-Realm", "POLARIS")
        .config("spark.sql.catalog.polaris.rest-metrics-reporting-enabled", "false")
        .config("spark.sql.catalog.polaris.credential", os.environ["POLARIS_CLIENT_ID"] + ":" + os.environ["POLARIS_CLIENT_SECRET"])
        .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.polaris.s3.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.sql.catalog.polaris.s3.path-style-access", "true")
        .config("spark.sql.catalog.polaris.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.sql.catalog.polaris.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.sql.catalog.polaris.client.region", "us-east-1")
        .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential"))
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print("SPARK_STARTED", spark.version, spark.sparkContext.applicationId)
    return spark

from pyspark.sql import Window, functions as F
spark = start_spark("scd-" + DB)
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS polaris.{DB} LOCATION '{WAREHOUSE}'")


## Publish immutable capture batches

Upload through the driver using the existing S3 Java client. Compare existing object bytes before reusing a batch. Upload its manifest last. Only manifest-complete batches are read below. Keep one uploader and one pipeline writer active.


In [ ]:
import hashlib
import json

j = spark.sparkContext._jvm
credentials = j.com.amazonaws.auth.BasicAWSCredentials(os.environ["AWS_ACCESS_KEY_ID"], os.environ["AWS_SECRET_ACCESS_KEY"])
endpoint = j.com.amazonaws.client.builder.AwsClientBuilder.EndpointConfiguration(os.getenv("S3_ENDPOINT", "http://seaweedfs:8333"), "us-east-1")
client = (j.com.amazonaws.services.s3.AmazonS3ClientBuilder.standard()
    .withEndpointConfiguration(endpoint).withPathStyleAccessEnabled(True)
    .withCredentials(j.com.amazonaws.auth.AWSStaticCredentialsProvider(credentials)).build())
batches = sorted((LAB_DIR / "batches").glob("batch_*"))
assert batches, "Capture or copy batches first; check SCD_CAPTURE_DIR."
product_paths, order_paths = [], []
try:
    for batch in batches:
        manifest = json.loads((batch / "manifest.json").read_text())
        assert manifest["database"] == DB, "Use the capture's database name in every notebook."
        files = sorted(batch.glob("product_cdc_*.json")) + sorted(batch.glob("orders_*.csv"))
        assert len(files) == 2, "Each batch requires one product JSON and one order CSV."
        for file in files + [batch / "manifest.json"]:
            key = f"scd-capture/{DB}/{batch.name}/{file.name}"
            if client.doesObjectExist("warehouse", key):
                obj = client.getObject("warehouse", key)
                stream = obj.getObjectContent()
                try:
                    old = bytes(j.org.apache.commons.io.IOUtils.toByteArray(stream))
                finally:
                    stream.close()
                assert hashlib.sha256(old).digest() == hashlib.sha256(file.read_bytes()).digest(), "Existing raw object differs: " + key
            else:
                client.putObject("warehouse", key, j.java.io.File(str(file.resolve())))
        product_paths.append(BRONZE + "/" + batch.name + "/" + files[0].name)
        order_paths.append(BRONZE + "/" + batch.name + "/" + files[1].name)
        print("Published / verified", batch.name)
finally:
    client.shutdown()


## Bronze: inspect CDC images and the complete order-event history

Use explicit types: money is DECIMAL, identifiers are BIGINT and timestamps are UTC.
`before`/`after` remain in the raw JSON. Flat after/current fields simplify downstream SQL.

In [ ]:
product_image = "STRUCT<product_id: BIGINT, product_name: STRING, category: STRING, color: STRING, list_price: DECIMAL(10,2), updated_at: TIMESTAMP>"
product_schema = ("batch_id BIGINT, sequence BIGINT, cdc_id STRING, operation STRING, product_id BIGINT, "
    "product_name STRING, category STRING, color STRING, list_price DECIMAL(10,2), event_time TIMESTAMP, "
    "binlog_time TIMESTAMP, source_updated_at TIMESTAMP, before " + product_image + ", after " + product_image)
order_schema = ("event_id BIGINT, order_id BIGINT, product_id BIGINT, quantity INT, selling_price DECIMAL(10,2), "
                "status STRING, event_time TIMESTAMP, created_at TIMESTAMP")
raw_products = spark.read.schema(product_schema).json(product_paths)
orders = (spark.read.schema(order_schema).option("header", True).option("mode", "FAILFAST")
    .csv(order_paths))
raw_products.orderBy("sequence").show(truncate=False)
orders.orderBy("event_id").show(100, truncate=False)
assert raw_products.filter("cdc_id IS NULL OR product_id IS NULL OR event_time IS NULL").count() == 0
assert orders.filter("event_id IS NULL OR event_time IS NULL").count() == 0
assert raw_products.select("cdc_id").distinct().count() == raw_products.count()
assert orders.select("event_id").distinct().count() == orders.count()

## Silver: flatten and preserve every change

Do not deduplicate products to their latest row here: SCD2 needs intermediate changes,
including several updates to the same product in one captured batch.

In [ ]:
changes = raw_products.select("batch_id", "sequence", "cdc_id", "operation", "product_id",
    "product_name", "category", "color", "list_price", "event_time")
changes.orderBy("sequence").show(100, truncate=False)
changes.createOrReplaceTempView("incoming_changes")
orders.createOrReplaceTempView("incoming_orders")
spark.sql(f"CREATE TABLE IF NOT EXISTS polaris.{DB}.product_changes USING iceberg AS SELECT * FROM incoming_changes WHERE 1=0")
spark.sql(f"CREATE TABLE IF NOT EXISTS polaris.{DB}.order_history USING iceberg AS SELECT * FROM incoming_orders WHERE 1=0")
spark.sql(f"""MERGE INTO polaris.{DB}.product_changes t USING incoming_changes s
    ON t.cdc_id=s.cdc_id WHEN NOT MATCHED THEN INSERT *""")
spark.sql(f"""MERGE INTO polaris.{DB}.order_history t USING incoming_orders s
    ON t.event_id=s.event_id WHEN NOT MATCHED THEN INSERT *""")

## Resolve current order state by business key

Business time comes first; physical `event_id` breaks ties. Arrival time would incorrectly
turn order 1001 back to PAID after Change 3. A DELIVERED filter loses CANCELLED, PAID and CREATED
orders. Keep all events in `order_history` and exactly one row per order in `latest_orders`.

In [ ]:
history = spark.table(f"polaris.{DB}.order_history")
window = Window.partitionBy("order_id").orderBy(F.col("event_time").desc(), F.col("event_id").desc())
latest = history.withColumn("rn", F.row_number().over(window)).filter("rn=1").drop("rn")
latest.createOrReplaceTempView("resolved_orders")
spark.sql(f"CREATE TABLE IF NOT EXISTS polaris.{DB}.latest_orders USING iceberg AS SELECT * FROM resolved_orders WHERE 1=0")
spark.sql(f"""MERGE INTO polaris.{DB}.latest_orders t USING resolved_orders s
    ON t.order_id=s.order_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
history.groupBy("order_id").count().orderBy("order_id").show(truncate=False)
latest.orderBy("order_id").show(truncate=False)
history.filter("order_id=1001").orderBy("created_at", "event_id").show(truncate=False)
print("All current orders:", latest.count(), "| Delivered only:", latest.filter("status='DELIVERED'").count())

## SCD Type 1: overwrite the current representation

One source row per product is required for MERGE. Resolve the latest CDC row by source
sequence; retain DELETE rows until MERGE so an older INSERT cannot resurrect a deleted product.
This lab replays all immutable changes, so a rerun computes the same current state.

In [ ]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{DB}.dim_product_scd1 (
    product_id BIGINT, product_name STRING, category STRING, color STRING,
    list_price DECIMAL(10,2), updated_at TIMESTAMP) USING iceberg""")
product_window = Window.partitionBy("product_id").orderBy(F.col("sequence").desc())
current_changes = (spark.table(f"polaris.{DB}.product_changes")
    .withColumn("rn", F.row_number().over(product_window)).filter("rn=1").drop("rn"))
current_changes.createOrReplaceTempView("current_product_changes")
spark.sql(f"""MERGE INTO polaris.{DB}.dim_product_scd1 t USING current_product_changes s
    ON t.product_id=s.product_id
    WHEN MATCHED AND s.operation='D' THEN DELETE
    WHEN MATCHED AND s.operation<>'D' THEN UPDATE SET
        t.product_name=s.product_name, t.category=s.category, t.color=s.color,
        t.list_price=s.list_price, t.updated_at=s.event_time
    WHEN NOT MATCHED AND s.operation<>'D' THEN INSERT
        (product_id,product_name,category,color,list_price,updated_at)
        VALUES(s.product_id,s.product_name,s.category,s.color,s.list_price,s.event_time)""")
spark.table(f"polaris.{DB}.dim_product_scd1").orderBy("product_id").show(truncate=False)

## Price-period analysis without double-counting lifecycle events

Use actual selling price, not today's list price. The measure below is **booked value for
currently PAID or DELIVERED orders**, not accounting revenue. CANCELLED and CREATED are excluded
only from this measure, never from latest-order resolution. One lifecycle row per order
prevents counting the same order once for CREATED and again for PAID.

In [ ]:
spark.sql(f"""SELECT selling_price, COUNT(*) AS orders, SUM(quantity) AS quantity,
    SUM(quantity * selling_price) AS booked_value
    FROM polaris.{DB}.latest_orders WHERE status IN ('PAID','DELIVERED')
    GROUP BY selling_price ORDER BY selling_price DESC""").show(truncate=False)
assert latest.count() == latest.select("order_id").distinct().count()
print("Next: run notebook 3 to preserve product history.")

## Release Spark

Stop this session before starting the next notebook.

In [ ]:
spark.stop()
